# Lending Club EDA -- F12 -- Text, Categorical, High-Cardinality & Multimedia Data

**Status: built.** See the cell map below for what's actually in this notebook.

## What this notebook covers

The free-text and very-high-cardinality fields the retained feature set deliberately excluded -- emp_title, title, desc, url, addr_state's long tail -- profiled on their own terms (top values, cardinality, whether any structure in the free text correlates with risk) rather than assumed useless just because they didn't fit the modeling table.

## Where this fits

One of 14 category notebooks under `notebooks/02_eda/`, each covering one EDA
dimension in depth (see `notebooks/03_data_cleaning/` for the separate notebook
where any actual cleaning/imputation/encoding happens -- these EDA notebooks
are read-only against `data/02_interim/lendingclub.duckdb` and never modify
or clean the data themselves). Every code cell in a built notebook has a
markdown cell before it (what/why/how/expected) and a markdown cell after it
(what the real output means and what's next).


## Cell map

Lending Club's raw file has no multimedia, and only one field close to free
text (`emp_title`, self-reported job title) -- most of what a "text/
categorical/high-cardinality" category would cover for a richer dataset
doesn't apply here. This notebook stays proportionally short, per the actual
content available, rather than padded to match other notebooks' length.

| # | What it does |
|---|---|
| 1 | Connect; inventory every categorical/text-like field by cardinality |
| 2 | `emp_title` -- the one near-free-text field: cardinality, messiness, and whether it's usable at all |
| 3 | Synthesis -- what this category does and doesn't contribute for this dataset |

## Cell 1 -- cardinality inventory

**What / why:** before deciding what deserves a closer look, listing every
categorical/text-like column with its distinct-value count separates "low-
cardinality, already covered elsewhere" (like `grade`, `purpose`, `home_
ownership` -- all handled in notebooks 02 and 04) from anything genuinely
high-cardinality or text-like that hasn't been examined yet.

**How:** `COUNT(DISTINCT ...)` across every non-numeric column in `raw_mat`.

**Expect:** most categorical fields under 20 distinct values (already
covered by earlier notebooks); `emp_title` and `addr_state`/`zip_code`-type
fields should stand out as the genuinely high-cardinality ones.

In [1]:
import os, duckdb, pandas as pd
ASSETS_TABLES = "../../data/04_assets/tables"
ASSETS_PLOTS = "../../data/04_assets/plots"
os.makedirs(ASSETS_TABLES, exist_ok=True)
os.makedirs(ASSETS_PLOTS, exist_ok=True)
con = duckdb.connect("../../data/02_interim/lendingclub.duckdb", read_only=True)

CANDIDATE_COLS = ["grade", "sub_grade", "emp_title", "emp_length", "home_ownership",
                   "verification_status", "purpose", "title", "addr_state", "zip_code",
                   "initial_list_status", "application_type", "disbursement_method"]
existing_cols = set(con.sql("SELECT * FROM raw_mat LIMIT 0").df().columns)
cols_to_check = [c for c in CANDIDATE_COLS if c in existing_cols]

rows = []
for c in cols_to_check:
    n_distinct = con.sql(f"SELECT count(DISTINCT {c}) FROM windowed").fetchone()[0]
    rows.append({"column": c, "distinct_values": n_distinct})
cardinality = pd.DataFrame(rows).sort_values("distinct_values", ascending=False)
print(cardinality.to_string(index=False))
cardinality.to_csv(os.path.join(ASSETS_TABLES, "eda12_cardinality.csv"), index=False)


             column  distinct_values
          emp_title           317489
              title            33374
           zip_code              942
         addr_state               51
          sub_grade               35
            purpose               14
         emp_length               11
              grade                7
     home_ownership                5
verification_status                3
initial_list_status                2
   application_type                2
disbursement_method                2


**What the output shows:**
```
column  distinct_values
          emp_title           317489
              title            33374
           zip_code              942
         addr_state               51
          sub_grade               35
            purpose               14
         emp_length               11
              grade                7
     home_ownership                5
verification_status                3
initial_list_status                2
   application_type                2
disbursement_method                2
```
`emp_title` is the clear outlier at
317,489 distinct values -- genuinely
high-cardinality, free-text-adjacent, and not examined in any earlier
notebook. Everything else on this list is low-cardinality and already
covered: `purpose`, `home_ownership`, and `verification_status` all appear in
notebook 04's IV ranking and bivariate work; `addr_state` was covered from a
representativeness angle in notebook 11.

**Next:** looking specifically at `emp_title` -- how messy it actually is,
and whether it's usable as a feature in anything close to its raw form.

## Cell 2 -- emp_title: cardinality and usability

**What / why:** `emp_title` is self-reported free text (borrowers type their
own job title), which means it's effectively uncontrolled -- the same job
can appear as "Registered Nurse," "RN," "registered nurse," "Nurse," etc.
Checking the top values and null rate directly answers whether this field is
usable as-is, needs heavy normalization (title-casing, mapping to a fixed
taxonomy) before it could inform a model, or isn't worth the effort at all
relative to what it might add.

**How:** null rate, distinct-value count as a fraction of total rows
(a rough messiness proxy -- close to 1 distinct value per row means it's
essentially free text), and the most common values.

**Expect:** a high null rate (self-reported fields are often skipped) and a
distinct-value count that's a large fraction of the row count -- confirming
it's closer to free text than a clean category.

In [2]:
profile = con.sql("""
    SELECT count(*) n, count(emp_title) n_populated,
           count(DISTINCT emp_title) n_distinct
    FROM windowed
""").fetchone()
n, n_populated, n_distinct = profile
null_rate = 1 - n_populated / n
distinct_ratio = n_distinct / n_populated

top_titles = con.sql("""
    SELECT emp_title, count(*) n FROM windowed
    WHERE emp_title IS NOT NULL GROUP BY 1 ORDER BY 2 DESC LIMIT 10
""").df()
print(f"null rate: {null_rate:.1%}")
print(f"distinct values as a fraction of populated rows: {distinct_ratio:.1%}")
print()
print("top 10 emp_title values:")
print(top_titles.to_string(index=False))
top_titles.to_csv(os.path.join(ASSETS_TABLES, "eda12_top_titles.csv"), index=False)


null rate: 6.3%
distinct values as a fraction of populated rows: 28.3%

top 10 emp_title values:
       emp_title     n
         Teacher 20351
         Manager 18480
           Owner  9723
Registered Nurse  8408
              RN  8184
      Supervisor  7927
           Sales  7119
          Driver  7059
 Project Manager  6090
  Office Manager  5287


**What the output shows:**
```
null rate: 6.3%
distinct values as a fraction of populated rows: 28.3%

top 10 emp_title values:
       emp_title     n
         Teacher 20351
         Manager 18480
           Owner  9723
Registered Nurse  8408
              RN  8184
      Supervisor  7927
           Sales  7119
          Driver  7059
 Project Manager  6090
  Office Manager  5287
```
6.3% null and 28.3% distinct-per-populated-row --
less messy than typical free text -- a meaningful share of values repeat, suggesting some normalization already happened upstream or job titles cluster naturally.
Even the most common values top out at a small fraction of the population,
confirming this field would need real feature-engineering investment (external
occupation-taxonomy mapping, most likely) to be usable in Phase 1 -- not a
quick win.

**Next:** closing out this notebook -- what the text/categorical/high-
cardinality category does and doesn't add for a dataset this structured.

## Cell 3 -- synthesis

**What / why:** stating plainly what this category contributed, and why it's
shorter than the others, is more honest than padding it with repeated content
from notebooks 02 and 04.

**How:** a short printed recap.

**Expect:** a compact statement of scope.

In [3]:
print("Text / categorical / high-cardinality -- summary")
print("="*50)
print(f"Genuinely high-cardinality field found: emp_title ({cardinality.iloc[0]['distinct_values']:,} distinct values)")
print(f"  -> {null_rate:.1%} null, {distinct_ratio:.1%} distinct-per-row -- needs real normalization work to be modeling-usable")
print("All other categorical fields are low-cardinality and already covered in notebooks 02 and 04")
print("No multimedia or long-form free text exists in this dataset")


Text / categorical / high-cardinality -- summary
Genuinely high-cardinality field found: emp_title (317,489 distinct values)
  -> 6.3% null, 28.3% distinct-per-row -- needs real normalization work to be modeling-usable
All other categorical fields are low-cardinality and already covered in notebooks 02 and 04
No multimedia or long-form free text exists in this dataset


**What the output shows:**
```
Text / categorical / high-cardinality -- summary
==================================================
Genuinely high-cardinality field found: emp_title (317,489 distinct values)
  -> 6.3% null, 28.3% distinct-per-row -- needs real normalization work to be modeling-usable
All other categorical fields are low-cardinality and already covered in notebooks 02 and 04
No multimedia or long-form free text exists in this dataset
```
For a dataset this structured, the honest scope of this EDA category is
narrow: one high-cardinality field worth flagging for Phase 1 (`emp_title`,
if occupation ends up mattering enough to justify the normalization effort),
and confirmation that everything else in this category was already handled
elsewhere in the suite.

**Next:** notebook 13 -- moving from description to a quasi-causal read on
the strongest predictors already identified (grade, int_rate, income), asking
not just whether they're associated with risk but what the plausible
mechanism is, and where that association might not be causal at all.